# Isolation Forest — Network Anomaly Detection

Used by **Scenario 11** (AI Anomaly Detection), **Scenario 15** (DNS tunneling features),
and **Scenario 23** (retrain on realistic DVWA/Juice Shop traffic).

Trains an unsupervised Isolation Forest on a 'normal' baseline of network-connection
features, then flags slow/stealthy outliers (e.g. low-and-slow C2 beaconing) that
signature rules miss.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(42)

## 1. Build a normal baseline
In the real lab, replace this synthetic data with 7 days of Sysmon EventID 3
(network connections) exported from Splunk. Features: bytes out, connection
interval (s), destination port entropy.

In [ ]:
n = 5000
normal = pd.DataFrame({
    'bytes_out': rng.normal(1500, 400, n).clip(0),
    'interval_s': rng.normal(2, 0.5, n).clip(0),
    'dport_entropy': rng.normal(3.5, 0.4, n).clip(0),
})
normal.describe()

In [ ]:
model = IsolationForest(n_estimators=200, contamination=0.01, random_state=42)
model.fit(normal)
print('Model trained on', len(normal), 'baseline samples')

## 2. Inject a low-and-slow C2 beacon
A beacon sends a tiny request on a fixed long interval — abnormal on every axis.

In [ ]:
beacon = pd.DataFrame({
    'bytes_out': [120, 118, 125, 119],
    'interval_s': [300, 300, 301, 299],
    'dport_entropy': [0.2, 0.2, 0.3, 0.2],
})
test = pd.concat([normal.sample(20, random_state=1), beacon], ignore_index=True)
test['anomaly'] = model.predict(test[['bytes_out', 'interval_s', 'dport_entropy']])
test[test['anomaly'] == -1]   # -1 == flagged anomaly

## 3. Push anomalies to Splunk HEC (optional)
Forward each `-1` row as a JSON alert so the blue team sees it in `index=ml_alerts`.

In [ ]:
import os, requests, urllib3
urllib3.disable_warnings()
HEC = os.getenv('SPLUNK_HEC', 'https://localhost:8088/services/collector')
TOKEN = os.getenv('SPLUNK_TOKEN', '')
for _, row in test[test['anomaly'] == -1].iterrows():
    event = {'sourcetype': 'isolation_forest', 'index': 'ml_alerts', 'event': {
        'anomaly_score': -1, 'bytes_out': float(row['bytes_out']),
        'interval_s': float(row['interval_s']),
        'anomaly_reason': 'low-and-slow beacon'}}
    if TOKEN:
        try:
            requests.post(HEC, headers={'Authorization': f'Splunk {TOKEN}'},
                          json=event, verify=False, timeout=10)
        except requests.RequestException as e:
            print('HEC forward failed:', e)
    else:
        print('would forward:', event)